# insurance-distributional-glm

**GAMLSS for insurance pricing in Python.**

Standard GLMs model `E[Y|X]` — the conditional mean. That is fine when you
believe every risk with the same mean also has the same variance. But in
motor insurance, a young driver and a middle-aged driver with identical expected
claims can have dramatically different claim volatility.

GAMLSS fixes this by modelling the full conditional distribution `p(Y|X)`,
not just its mean. Each distribution parameter — mean, variance, shape —
is expressed as a function of covariates.

R has had this since 2005 (`gamlss` package). Python has had nothing production-ready.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-distributional-glm/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q "insurance-distributional-glm[plots]" polars numpy

## Synthetic severity data with heterogeneous variance

We generate 4,000 motor claims. The mean severity depends on vehicle age
and NCD years (standard GLM territory). But the variance — the coefficient
of variation — also depends on vehicle age: older vehicles have more volatile
repair costs (large variance when they do claim).

A standard GLM with Gamma family cannot see this. `DistributionalGLM` can.

In [ ]:
import numpy as np
import polars as pl

rng = np.random.default_rng(42)
n = 4_000

# Covariates
vehicle_age = rng.integers(0, 15, n).astype(float)
ncd_years   = rng.integers(0, 9, n).astype(float)
area        = rng.integers(0, 4, n).astype(float)

# True mean: depends on all three
log_mu = 6.5 + 0.05 * vehicle_age - 0.04 * ncd_years + 0.1 * area
mu = np.exp(log_mu)   # mean severity ~£650

# True dispersion: OLD vehicles have higher sigma (more variable repairs)
# Young NCD policyholders also have higher variance (fewer claims, more extreme)
log_sigma = -1.0 + 0.08 * vehicle_age - 0.05 * ncd_years
sigma = np.exp(log_sigma)  # CV of Gamma distribution

# Gamma parameterisation: shape = 1/sigma^2, scale = mu*sigma^2
shape = 1.0 / sigma**2
scale = mu * sigma**2
severity = rng.gamma(shape=shape, scale=scale)

df = pl.DataFrame({
    "vehicle_age": vehicle_age,
    "ncd_years":   ncd_years,
    "area":        area,
})

print(f"{n:,} claims | mean severity: £{severity.mean():.0f}")
print(f"True sigma range: [{sigma.min():.3f}, {sigma.max():.3f}]")
df.head(4)

## Fit a standard Gamma GLM (mean-only)

First, the standard approach: a Gamma GLM modelling only the mean.
This is correct for the mean parameter but assumes constant sigma — it
cannot see the vehicle-age effect on variance.

In [ ]:
from insurance_distributional_glm import DistributionalGLM
from insurance_distributional_glm.families import Gamma

# Standard GLM: only model mu
glm_standard = DistributionalGLM(
    family=Gamma(),
    formulas={"mu": ["vehicle_age", "ncd_years", "area"], "sigma": []},
)
glm_standard.fit(df, severity)

print("Standard Gamma GLM (mean only):")
print(f"  Log-likelihood: {glm_standard.loglik_:.2f}")
print(f"  AIC:            {glm_standard.aic_:.2f}")
print(f"  mu coefficients: {dict(zip(['intercept','vehicle_age','ncd_years','area'], [round(c,4) for c in glm_standard.coefs_['mu']]))}")
print(f"  sigma (const):  {glm_standard.sigma_:.4f}")

## Fit the distributional GLM (mu + sigma both modelled)

Now we let sigma depend on vehicle_age and ncd_years too.
The distributional GLM fits both equations simultaneously via CG iteration.

In [ ]:
glm_dist = DistributionalGLM(
    family=Gamma(),
    formulas={
        "mu":    ["vehicle_age", "ncd_years", "area"],
        "sigma": ["vehicle_age", "ncd_years"],
    },
)
glm_dist.fit(df, severity)

print("Distributional GLM (mu + sigma):")
print(f"  Log-likelihood: {glm_dist.loglik_:.2f}")
print(f"  AIC:            {glm_dist.aic_:.2f}")
print(f"  AIC improvement: {glm_standard.aic_ - glm_dist.aic_:.1f}")
print()
print("  sigma coefficients:")
for name, coef in zip(['intercept', 'vehicle_age', 'ncd_years'], glm_dist.coefs_['sigma']):
    print(f"    {name:<15}: {coef:.4f}")

## Distribution selection

`choose_distribution` fits several candidate families and ranks them by GAIC.
Use this when you are not sure whether Gamma, Lognormal, or Inverse-Gaussian
is the right marginal for your severity data.

In [ ]:
from insurance_distributional_glm import choose_distribution
from insurance_distributional_glm.families import Gamma, LogNormal, InverseGaussian

selection = choose_distribution(
    df, severity,
    candidates=[Gamma(), LogNormal(), InverseGaussian()],
    formula={"mu": ["vehicle_age", "ncd_years", "area"], "sigma": ["vehicle_age"]},
)

print("Distribution ranking by GAIC:")
for i, result in enumerate(selection.ranked):
    marker = " <-- best" if i == 0 else ""
    print(f"  {result.family:<20} AIC={result.aic:.1f}  GAIC={result.gaic:.1f}{marker}")

## What you should see

- The distributional GLM AIC should be materially lower than the standard GLM
  (at least 10 points, probably 50-100 given how strongly we designed the
  vehicle_age → sigma relationship).
- The sigma coefficient on vehicle_age should be positive (~0.08) — matching
  the true generative process.
- Gamma should rank first in distribution selection (we generated data from it).

## Next steps

- **`quantile_residuals()`** — randomised quantile residuals for distributional
  GLM diagnostics (worm plots)
- **`worm_plot()`** — visual diagnostic for distributional misspecification
- **`families.ZIP`** — Zero-Inflated Poisson for modelling structural zeros in
  frequency data
- **`families.NegativeBinomial`** — NB2 with modelled overdispersion

**GitHub:** https://github.com/burning-cost/insurance-distributional-glm  
**PyPI:** https://pypi.org/project/insurance-distributional-glm/